**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/churchland-2012"], check=True)
    os.chdir("/content/encore/entries/churchland-2012")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Churchland et al. (2012), "Neural population dynamics during reaching", from DANDI:000070

- Paper: Churchland MM, Cunningham JP, Kaufman MT, Foster JD, Nuyujukian P, Ryu SI, Shenoy KV (2012). Neural population dynamics during reaching. *Nature* 487, 51-56. doi:10.1038/nature11129.
- Claims: `claims/churchland-2012.yaml`, written from the NIH author manuscript in PubMed Central (PMC3393826), its Methods and figure legends, and the Supplementary Information PDF.
- Dataset: DANDI:000070, version 0.261122.2046 (doi:10.48324/dandi.000070/0.261122.2046). This version was asked for in the prompt of this attempt, a blind repeat attempt made after the dataset changed. The paper cites no version; the version asked for is also the most recent published one. All files are streamed from the archive; nothing is downloaded to disk.
- Sessions used: sub-Jenkins ses-20090918 (the paper's J-array, recorded 09-18-2009) and sub-Nitschke ses-20100923 (N-array, 09-23-2010), each with 108 maze conditions. sub-Nitschke ses-20100910 (108 conditions) is used as the "N-9-10-2010" dataset of Supplementary Figure 5, as a variant only.

Method choices are those of the claims file (its header) and, where it is silent, the paper, cited in the helper `reach.py` and below. Two details the paper leaves open are run under each reading: which movement-onset column of the trial table is used (`offline_move_onset_time`, main, or `move_begins_time`), and the window over which a unit's firing-rate range is taken for the soft normalisation (the analysed window, main, or -1000 to +700 ms). The outcome is the one all four readings give; where they differ, the main reading is adopted as an assumption.

| Claim | Tested here |
|---|---|
| array-dataset-composition | tested |
| fig3ef-same-direction | tested |
| fig3-independent-of-curvature | tested |
| fig3-phase-from-prep-state | tested |
| var-captured-jpc-plane | tested |
| pc23-variance | tested |
| suppfig4-j-array-planes | tested |
| suppfig4-planes-range | tested |
| fig6a-angle-peak | tested |
| fig6b-skew-fit | tested |
| rotation-frequency | tested |
| suppfig3-shuffle-controls | tested |
| suppfig5-areas-variance | tested |
| suppfig5-m1-faster | tested |
| fig6-velocity-model | tested (simulation; rests on assumptions about the model) |
| fig6-complex-kinematic-model | tested (simulation; rests on assumptions about the model) |
| fig1-leech-and-walking | blocked: leech and walking-monkey data are not in the dandiset |
| fig2-single-neuron-responses | blocked: single-electrode datasets are not in the dandiset |
| fig3abcd-single-electrode-datasets | blocked: single-electrode datasets are not in the dandiset |
| fig4-emg-no-rotation | blocked: no EMG in the dandiset |
| fig5-generator-model | blocked: no EMG in the dandiset |
| fig6cd-speed-independent-frequency | blocked: the speed-task datasets (monkeys A and B) are not in the dandiset |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import time, zlib, hashlib, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import reach as R

KEY = "churchland-2012"
def seed(claim, extra=0):
    return zlib.crc32(f"{KEY}/{claim}".encode()) + extra
SEED_OFFSETS = [0, 1000, 2000, 3000, 4000]
T_START = time.time()
print("dandiset", R.DANDISET, "version", R.VERSION)

## Data check

The archive's metadata and every file of the version: trials, conditions, units and spikes. Duplicate deposits are looked for by hashing each file's trial start times and its spike counts per unit.

In [ ]:
meta = R.version_metadata()
print(meta["name"], "|", meta["version"], "|", meta.get("doi"))
print("description:", meta.get("description"))
print("subjects:", meta["assetsSummary"].get("numberOfSubjects"), "files:", meta["assetsSummary"].get("numberOfFiles"))
assets = R.list_assets()
rows = []
for a in assets:
    f = R.open_asset(a["path"])
    tr = f["intervals/trials"]
    st = tr["start_time"][:]
    si = f["units/spike_times_index"][:]
    row = {"path": a["path"].split("/")[-1], "GB": a["size"] / 1e9, "trials": len(st),
           "has_maze_condition": "maze_condition" in tr, "units": len(si),
           "spikes": f["units/spike_times"].shape[0],
           "start_hash": hashlib.md5(np.round(np.diff(st), 4).tobytes()).hexdigest()[:10],
           "units_hash": hashlib.md5(np.diff(np.r_[0, si]).tobytes()).hexdigest()[:10]}
    if "maze_condition" in tr:
        d = R.load_trials(f)
        sel = R.select_trials(d)
        row["conditions"] = int(d.maze_condition[d.maze_condition > 0].nunique())
        row["selected_trials"] = int(sel.sum())
        row["selected_conditions"] = int(d.maze_condition[sel].nunique())
    rows.append(row)
    f.close()
survey = pd.DataFrame(rows)
print(survey.to_string())
print("distinct trial-interval hashes:", survey.start_hash.nunique(), "of", len(survey),
      "| distinct spike-count hashes:", survey.units_hash.nunique(), "of", len(survey))

Load the three sessions used: trial table, condition-mean spike counts aligned on each movement-onset column, per-trial counts for the clock check, and hand position.

In [ ]:
DATA = {}
for name, path in R.SESSIONS.items():
    t0 = time.time()
    f = R.open_asset(path)
    d = R.load_trials(f)
    sel = R.select_trials(d)
    tr = d[sel].reset_index(drop=True)
    L = R.load_rates(f, tr)
    groups = R.unit_groups(f)
    gdesc = {g: f[f"general/extracellular_ephys/{g}"].attrs["description"] for g in np.unique(groups)}
    udesc = f["units"].attrs["description"]
    rates = {col: R.smooth_rates(L[col]["sum"], L[col]["n"]).astype(np.float32)
             for col in ("offline_move_onset_time", "move_begins_time")}
    rates24 = R.smooth_rates(L["offline_move_onset_time"]["sum"], L["offline_move_onset_time"]["n"], sd=24.0).astype(np.float32)
    hand = None
    if name != "N-9-10-2010":
        hand, th = R.load_hand(f, tr)
    onset_desc = {c: f[f"intervals/trials/{c}"].attrs.get("description") for c in ("offline_move_onset_time", "move_begins_time")}
    DATA[name] = dict(all_trials=d, trials=tr, n=L["offline_move_onset_time"]["n"], rates=rates, rates24=rates24,
                      groups=groups, gdesc=gdesc, udesc=udesc, hand=hand,
                      block_sum=L["block_sum"], block=L["block"],
                      target_counts=L["target_counts"], move_counts=L["move_counts"],
                      target_edges=L["target_edges"], move_edges=L["move_edges"],
                      unsorted=L["units_unsorted"], outside=L["outside_trial"], onset_desc=onset_desc,
                      event_sum=L["event_sum"], event_window=L["event_window"])
    f.close()
    del L
    print(f"{name}: loaded in {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
TH = th

In [ ]:
for name, D in DATA.items():
    d, tr = D["all_trials"], D["trials"]
    print(f"== {name}")
    print("units:", D["udesc"])
    print("unit groups:", {g: int((D["groups"] == g).sum()) for g in np.unique(D["groups"])}, D["gdesc"])
    print(f"trials {len(d)}; successful {int((d.task_success == 1).sum())}; consistent {int((d.correct_reach == 1).sum())}; "
          f"discarded {int((d.discard_trial == 1).sum())}; delay > 400 ms {int((d.delay > 0.4).sum())}; selected {len(tr)}")
    print(f"delay range {d.delay.min() * 1000:.0f} to {d.delay.max() * 1000:.0f} ms; selected {tr.delay.min() * 1000:.0f} to {tr.delay.max() * 1000:.0f} ms")
    n = D["n"]
    print(f"conditions with selected trials {int((n > 0).sum())}; trials per condition {n.min()} to {n.max()} (median {np.median(n):.0f})")
    go = (tr.go_cue_time - tr.offline_move_onset_time) * 1000
    print(f"go cue relative to offline movement onset: median {go.median():.0f} ms (5-95%: {go.quantile(.05):.0f} to {go.quantile(.95):.0f})")
    dd = (tr.offline_move_onset_time - tr.move_begins_time) * 1000
    for c, desc in D["onset_desc"].items():
        print(f"description of {c}: {desc!r}")
    print(f"offline_move_onset_time minus move_begins_time: median {dd.median():.1f} ms (5-95%: {dd.quantile(.05):.1f} to {dd.quantile(.95):.1f})")
    print("units with unsorted spike times:", D["unsorted"], "| trials whose analysis window leaves the trial interval:", D["outside"])
    r = D["rates"]["offline_move_onset_time"]
    print(f"condition-mean rates (spikes/s): units' mean {r.mean(axis=(1, 2)).min():.1f} to {r.mean(axis=(1, 2)).max():.1f}; overall max {r.max():.0f}")
    if D["hand"] is not None:
        h = D["hand"]["offline_move_onset_time"]
        print(f"hand position (mm): x {np.nanmin(h[..., 0]):.0f} to {np.nanmax(h[..., 0]):.0f}, y {np.nanmin(h[..., 1]):.0f} to {np.nanmax(h[..., 1]):.0f}; trials without hand samples {int(np.isnan(h).all(axis=(1, 2)).sum())}")
        spd = np.linalg.norm(np.gradient(h, axis=1), axis=2) * 1000
        print("median hand speed (mm/s) at", {int(t): round(float(np.nanmedian(spd[:, TH == t])), 1) for t in (-100, -50, 0, 50, 100, 200, 400)}, "ms from offline onset")
        spd_b = np.linalg.norm(np.gradient(D["hand"]["move_begins_time"], axis=1), axis=2) * 1000
        print("median hand speed (mm/s) at", {int(t): round(float(np.nanmedian(spd_b[:, TH == t])), 1) for t in (-100, -50, 0, 50, 100, 200, 400)}, "ms from move_begins_time")
        end = h[:, TH == 450, :][:, 0, :]
        tgt = np.c_[tr.hit_x - tr.cursor_dx, tr.hit_y - tr.cursor_dy]
        err = np.linalg.norm(end - tgt, axis=1)
        print(f"hand at +450 ms against hit target minus cursor offset: median distance {np.nanmedian(err):.1f} mm (target positions span {tr.hit_x.min():.0f} to {tr.hit_x.max():.0f} mm in x)")

### Clock check

The spikes are saved per trial. Both arrays should share the clock of the trial table (the files say that the LFP of one array was recorded on a second, slower clock and mapped onto the first). The check decodes the reach direction (eight sectors of the hit target's direction) from each array's spike counts: in the 500 ms before target onset it should be at chance, and after target onset (100 to 400 ms) above it. Around movement onset, the time course of decoding and of the across-condition spread of rates is compared between arrays: an offset between the arrays would show as a shift. The same comparison is made after target onset and after the go cue, events that do not depend on the movement onset: a fixed offset between the arrays' spike clocks would shift the responses to every event by the same amount, while a later M1 response around movement would not.

In [ ]:
CLOCK = {}
for name in ("J-array", "N-array"):
    D = DATA[name]
    tr = D["trials"]
    lab = R.reach_direction_class(tr)
    rng = np.random.default_rng(seed("clock-check"))
    tc, mc = D["target_counts"], D["move_counts"]
    te, me = D["target_edges"], D["move_edges"]
    pre = (te >= -0.5) & (te < 0)
    post = (te >= 0.1) & (te < 0.4)
    for g in np.unique(D["groups"]):
        m = D["groups"] == g
        a_pre = R.decode_accuracy(tc[:, m][:, :, pre].sum(2).astype(float), lab, rng)
        a_post = R.decode_accuracy(tc[:, m][:, :, post].sum(2).astype(float), lab, rng)
        a_null = R.decode_accuracy(tc[:, m][:, :, post].sum(2).astype(float), rng.permutation(lab), rng)
        acc = np.array([R.decode_accuracy(mc[:, m, b].astype(float), lab, rng) for b in range(len(me))])
        CLOCK[(name, g)] = acc
        print(f"{name} group {g}: before target {a_pre:.3f}, after target {a_post:.3f}, after target with shuffled labels {a_null:.3f} "
              f"(class sizes {np.bincount(lab, minlength=8).tolist()})")
    t = np.arange(R.T0, R.T1)
    r = D["rates"]["offline_move_onset_time"]
    spread = {g: r[D["groups"] == g].std(axis=1).mean(axis=0) for g in np.unique(D["groups"])}
    w = (t >= -400) & (t <= 400)
    a, b = np.diff(spread["1"][w]), np.diff(spread["2"][w])
    a, b = (a - a.mean()) / a.std(), (b - b.mean()) / b.std()
    lags = np.arange(-60, 61)
    xc = [np.mean(a[max(0, -l):len(a) - max(0, l)] * b[max(0, l):len(b) - max(0, -l)]) for l in lags]
    print(f"{name}: lag of group 2 behind group 1 at the peak cross-correlation of the change of across-condition spread: {lags[int(np.argmax(xc))]} ms (peak r {max(xc):.2f})")
    half = {g: t[w][np.argmax(spread[g][w] >= spread[g][w].min() + 0.5 * (spread[g][w].max() - spread[g][w].min()))] for g in spread}
    print(f"{name}: time at which the across-condition spread reaches half its rise: group 1 {half['1']} ms, group 2 {half['2']} ms")
from scipy.ndimage import gaussian_filter1d
print("Drift check: the lag of group 2 behind group 1 in six consecutive blocks of each session, from the change of the")
print("grand-mean rate (10 ms Gaussian) from -400 to +300 ms around movement onset. An unmapped clock running 9.4 ppm slow")
print("(the LFP's second system: 999.9906 against 1000 samples/s) would add about 9.4 ms per 1000 s of session.")
for name in ("J-array", "N-array"):
    D = DATA[name]
    t = np.arange(R.T0 - R.PAD, R.T1 + R.PAD)
    w = (t >= -400) & (t <= 300)
    lags = np.arange(-150, 151)
    out = []
    for k in range(D["block_sum"].shape[1]):
        nk = int((D["block"] == k).sum())
        sig = []
        for g in ("1", "2"):
            x = gaussian_filter1d(D["block_sum"][D["groups"] == g, k].mean(0) / nk * 1000, 10)
            dx = np.diff(x[w]); sig.append((dx - dx.mean()) / dx.std())
        xc = [np.mean(sig[0][max(0, -l):len(sig[0]) - max(0, l)] * sig[1][max(0, l):len(sig[1]) - max(0, -l)]) for l in lags]
        t_start = D["trials"].start_time.values[D["block"] == k].min()
        out.append((round(t_start), int(lags[int(np.argmax(xc))]), round(max(xc), 2)))
    print(f"{name}: (block start s, lag ms, peak r):", out)
print("Latency of each array on events that do not depend on the movement onset: the across-condition spread of the")
print("condition-mean rates (20 ms Gaussian, mean over the array's units) after target onset and after the go cue. A fixed")
print("offset between the arrays' spike clocks would shift every event by the same amount.")
for name in ("J-array", "N-array"):
    D = DATA[name]
    ew = D["event_window"]
    te_ms = np.arange(ew[0], ew[1])
    for ev, ev_lab, win in (("target_presentation_time", "target onset", (-100, 300)), ("go_cue_time", "go cue", (-100, 400))):
        mean = D["event_sum"][ev] / np.maximum(D["n"], 1)[None, :, None] * 1000.0
        smr = gaussian_filter1d(mean, 20, axis=2, mode="nearest")
        w = (te_ms >= win[0]) & (te_ms <= win[1])
        spread = {g: smr[D["groups"] == g].std(axis=1).mean(axis=0)[w] for g in ("1", "2")}
        a, b = np.diff(spread["1"]), np.diff(spread["2"])
        a, b = (a - a.mean()) / a.std(), (b - b.mean()) / b.std()
        lg = np.arange(-60, 61)
        xc = [np.mean(a[max(0, -l):len(a) - max(0, l)] * b[max(0, l):len(b) - max(0, -l)]) for l in lg]
        half = {g: te_ms[w][np.argmax(spread[g] >= spread[g].min() + 0.5 * (spread[g].max() - spread[g].min()))] for g in spread}
        print(f"{name} | {ev_lab} ({win[0]} to {win[1]} ms): lag of group 2 behind group 1 at the peak cross-correlation {lg[int(np.argmax(xc))]} ms (peak r {max(xc):.2f}); "
              f"half-rise of the spread: group 1 {half['1']} ms, group 2 {half['2']} ms")
fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for i, name in enumerate(("J-array", "N-array")):
    for g in ("1", "2"):
        ax[i].plot(DATA[name]["move_edges"] * 1000 + 25, CLOCK[(name, g)], label={"1": "PMd (group 1)", "2": "M1 (group 2)"}[g])
    ax[i].axvline(0, color="k", lw=0.5); ax[i].set_title(name); ax[i].set_xlabel("ms from movement onset")
ax[0].set_ylabel("decoding accuracy (8 directions)"); ax[0].legend()
plt.show()
print("plotted accuracies range", round(min(v.min() for v in CLOCK.values()), 3), "to", round(max(v.max() for v in CLOCK.values()), 3))

## The jPCA analysis

Every claim below uses the same pipeline (helper `reach.py`): condition-mean rates aligned on movement onset (Gaussian 20 ms SD), soft normalisation (range + 5 spikes/s), the cross-condition mean subtracted, PCA on -50 to +150 ms every 10 ms, and jPCA on the top 6 PCs (10 for Supplementary Figure 4). It is run under the four readings of the two open details.

In [ ]:
READINGS = [("offline_move_onset_time", None), ("offline_move_onset_time", (-1000, 700)),
            ("move_begins_time", None), ("move_begins_time", (-1000, 700))]
def rlabel(rd):
    return ("offline onset" if rd[0] == "offline_move_onset_time" else "move_begins") + ", range " + ("analysed window" if rd[1] is None else "-1000..+700 ms")
MAIN = READINGS[0]

def summarise(o):
    Z = o["planes"][0]
    nr = R.net_rotation(Z)
    pk, _, _ = R.angle_peak(R.state_derivative_angles(Z))
    rad_s, hz = R.freq_hz(np.abs(o["eig"]).max())
    return {"plane_var": 100 * o["plane_var"][0], "pc23_var": 100 * o["pc_var"][1:3].sum(), "top6_var": 100 * o["pc_var"].sum(),
            "ccw_frac": (nr > 0).mean(), "R2_M": o["R2_M"], "R2_skew": o["R2_skew"], "ratio_6d": o["R2_skew"] / o["R2_M"],
            "R2_M_plane": o["R2_M_plane"], "R2_skew_plane": o["R2_skew_plane"], "ratio_plane": o["R2_skew_plane"] / o["R2_M_plane"],
            "peak_over_pi": pk / np.pi, "max_eig_rad_s": rad_s, "freq_hz": hz}

RES = {}
for name in ("J-array", "N-array"):
    for rd in READINGS:
        X = R.preprocess(DATA[name]["rates"][rd[0]], rd[1])
        RES[(name, rd)] = R.jpca(X, 6)
    X = R.preprocess(DATA[name]["rates24"], None)
    RES[(name, "smooth24")] = R.jpca(X, 6)
TAB = pd.DataFrame({(n, rlabel(rd) if rd != "smooth24" else "24 ms Gaussian (variant)"): summarise(o) for (n, rd), o in RES.items()}).T
pd.set_option("display.width", 250)
print(TAB.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
lims = []
for i, name in enumerate(("J-array", "N-array")):
    Z = RES[(name, MAIN)]["planes"][0]
    col = plt.cm.RdYlGn((Z[0, :, 0] - Z[0, :, 0].min()) / np.ptp(Z[0, :, 0]))
    for c in range(Z.shape[1]):
        ax[i].plot(Z[0, c], Z[1, c], color=col[c], lw=0.8)
        ax[i].plot(Z[0, c, 0], Z[1, c, 0], "o", color=col[c], ms=3)
    ax[i].set_title(f"{name}: first jPCA plane, -50 to +150 ms"); ax[i].set_xlabel("jPC1"); ax[i].set_ylabel("jPC2"); ax[i].set_aspect("equal")
    print(f"{name}: jPC1 range {Z[0].min():.3f} to {Z[0].max():.3f}, jPC2 range {Z[1].min():.3f} to {Z[1].max():.3f} (all points shown)")
plt.show()

## array-dataset-composition

> The two array datasets, monkey J-array recorded 2009-09-18 and monkey N-array recorded 2010-09-23, each present 108 reach conditions and comprise 146 and 218 isolations (a mix of single and multi-unit isolations).

Criterion: conditions are the distinct `maze_condition` values above 0 among the trials the Methods select; isolations are the rows of the Units table. Reproduced when both sessions are in the dandiset with all four values within tolerance (108, 108, 146 +- 14.6, 218 +- 21.8); partially reproduced when the sessions and condition counts match but a unit count does not.

In [ ]:
exp = {"J-array": (108, 146, 14.6), "N-array": (108, 218, 21.8)}
comp = {}
for name in ("J-array", "N-array"):
    D = DATA[name]
    nc = int(D["trials"].maze_condition.nunique())
    nu = len(D["groups"])
    comp[name] = (nc, nu)
    print(f"{name} ({R.SESSIONS[name]}): conditions {nc} (expected 108), units {nu} (expected {exp[name][1]} +- {exp[name][2]}; "
          f"{'within' if abs(nu - exp[name][1]) <= exp[name][2] else 'outside'} tolerance)")
print("Units table description:", DATA["J-array"]["udesc"])
print("conditions match:", all(comp[n][0] == 108 for n in comp),
      "| unit counts within tolerance:", [abs(comp[n][1] - exp[n][1]) <= exp[n][2] for n in comp])

**Result: partially reproduced.** Both sessions are in the dandiset and each has 108 conditions among the selected trials, but each Units table has 192 rows, outside the tolerance of 146 and of 218. The files describe these rows as unsorted threshold crossings, one per electrode, not the paper's sorted isolations.

## fig3ef-same-direction

> In the first jPCA plane the neural state rotates in the same direction for all conditions of the J-array and N-array datasets, including reaches in opposite directions.

Criterion: for each condition, the net change of the polar angle of its state in the oriented jPC plane from -50 to +150 ms. Reproduced when at least 90% of the 108 conditions rotate counter-clockwise in each dataset; partially reproduced when at least 75% do in each, or 90% in one only. The shuffle controls (claim suppfig3-shuffle-controls) are shown as reference there.

In [ ]:
for name in ("J-array", "N-array"):
    for rd in READINGS + ["smooth24"]:
        o = RES[(name, rd)]
        nr = R.net_rotation(o["planes"][0])
        lab = rlabel(rd) if rd != "smooth24" else "24 ms Gaussian (variant)"
        print(f"{name} | {lab}: {int((nr > 0).sum())} of {len(nr)} conditions counter-clockwise ({100 * (nr > 0).mean():.1f}%); "
              f"median net rotation {np.degrees(np.median(nr)):.0f} deg (range {np.degrees(nr.min()):.0f} to {np.degrees(nr.max()):.0f})")
def outcome_dir(fr):
    if all(f >= 0.9 for f in fr): return "reproduced"
    if all(f >= 0.75 for f in fr) or any(f >= 0.9 for f in fr): return "partially_reproduced"
    return "not_reproduced"
for rd in READINGS:
    fr = [(R.net_rotation(RES[(n, rd)]["planes"][0]) > 0).mean() for n in ("J-array", "N-array")]
    print(rlabel(rd), "->", outcome_dir(fr))

**Result: reproduced.** Under the main reading 103 of 108 J-array conditions (95.4%) and 107 of 108 N-array conditions (99.1%) rotate counter-clockwise; under the four readings the shares are 94.4 to 96.3% and 99.1 to 100%, all at least 90%.

## fig3-independent-of-curvature

> The rotations do not relate directly to reach curvature; the neural state rotates the same way for clockwise-curving, counter-clockwise-curving and straight reaches.

Criterion: each condition's reach is classed by the net change of heading of its mean hand path from movement onset to +500 ms as clockwise (below -20 degrees), counter-clockwise (above +20 degrees) or straight. Reproduced when, in each dataset, at least 90% of the conditions in each class with at least five conditions rotate counter-clockwise; partially reproduced when this holds for every class at 75%. A logistic fit of rotation direction on curvature is reported as well.

The heading is undefined while the hand is at rest, at onset and at the end of the reach. The main measure takes the heading of the mean path over 5 to 20% of its length (from onset to +500 ms) and over 80 to 95% of it, and the net change between them. A variant takes the unwrapped heading of the velocity over the samples where speed exceeds 10% of its peak, from the first to the last of them. Both are choices made here (the claims file and the paper do not define the heading). Under each reading the hand paths are aligned on the same onset column as the neural data.

In [ ]:
from scipy.ndimage import gaussian_filter1d
def curvature(hand, tr, th):
    cm = tr.maze_condition.values
    Hm = np.stack([np.nanmean(hand[cm == c], 0) for c in range(1, 109)])
    Hs = gaussian_filter1d(Hm, 10, axis=1)
    w = (th >= 0) & (th < 500)
    chord, vel = [], []
    for c in range(108):
        p = Hs[c, w]
        d = np.r_[0, np.cumsum(np.linalg.norm(np.diff(p, axis=0), axis=1))]
        d /= d[-1]
        i1, i2, i3, i4 = [np.searchsorted(d, q) for q in (0.05, 0.2, 0.8, 0.95)]
        a1 = np.arctan2(*(p[i2] - p[i1])[::-1]); a2 = np.arctan2(*(p[i4] - p[i3])[::-1])
        chord.append(np.degrees(np.angle(np.exp(1j * (a2 - a1)))))
        v = np.gradient(p, axis=0); s = np.linalg.norm(v, axis=1)
        ok = s > 0.1 * s.max()
        hd = np.unwrap(np.arctan2(v[ok, 1], v[ok, 0]))
        vel.append(np.degrees(hd[-1] - hd[0]))
    return np.array(chord), np.array(vel), Hm

CURV = {}
for name in ("J-array", "N-array"):
    for col in ("offline_move_onset_time", "move_begins_time"):
        chord, vel, Hm = curvature(DATA[name]["hand"][col], DATA[name]["trials"], TH)
        if col == "offline_move_onset_time":
            DATA[name]["hand_mean"] = Hm
        CURV[(name, col)] = (chord, vel)
        nb = DATA[name]["trials"].groupby("maze_condition").maze_num_barriers.first().values
        print(f"{name} | hand aligned on {col}: heading change (main) range {chord.min():.0f} to {chord.max():.0f} deg; conditions without barriers: median |change| {np.median(np.abs(chord[nb == 0])):.0f} deg (n = {(nb == 0).sum()})")
    a, b = CURV[(name, "offline_move_onset_time")][0], CURV[(name, "move_begins_time")][0]
    print(f"{name}: conditions whose class differs between the two hand alignments: {int((np.sign(np.where(np.abs(a) > 20, a, 0)) != np.sign(np.where(np.abs(b) > 20, b, 0))).sum())}")

def classes(curv):
    return np.where(curv < -20, "clockwise", np.where(curv > 20, "counter-clockwise", "straight"))

def curv_outcome(per_ds):
    ok90 = all(all(f >= 0.9 for (n, f) in d.values() if n >= 5) for d in per_ds)
    ok75 = all(all(f >= 0.75 for (n, f) in d.values() if n >= 5) for d in per_ds)
    return "reproduced" if ok90 else ("partially_reproduced" if ok75 else "not_reproduced")

for meas, k in (("main (path segments)", 0), ("variant (velocity heading)", 1)):
    for rd in READINGS:
        per = []
        for name in ("J-array", "N-array"):
            cl = classes(CURV[(name, rd[0])][k])
            ccw = R.net_rotation(RES[(name, rd)]["planes"][0]) > 0
            d = {c: (int((cl == c).sum()), float(ccw[cl == c].mean()) if (cl == c).any() else np.nan) for c in ("clockwise", "straight", "counter-clockwise")}
            per.append(d)
            if rd == MAIN or k == 0:
                print(f"{meas} | {rlabel(rd)} | {name}: " + "; ".join(f"{c} n={n}, {100 * f:.1f}% CCW" for c, (n, f) in d.items()))
        print(f"   -> {meas} | {rlabel(rd)}: {curv_outcome(per)}")

for name in ("J-array", "N-array"):
    nr = R.net_rotation(RES[(name, MAIN)]["planes"][0])
    cv = CURV[(name, MAIN[0])][0]
    bad = np.where(nr <= 0)[0]
    print(f"{name}, main reading: conditions not rotating counter-clockwise: " + "; ".join(
        f"condition {c + 1} (net rotation {np.degrees(nr[c]):.0f} deg, heading change {cv[c]:.0f} deg, {classes(cv)[c]})" for c in bad))
    print(f"   heading change of all conditions: median {np.median(cv):.0f} deg; of the clockwise class: median {np.median(cv[cv < -20]):.0f} deg")
for name in ("J-array", "N-array"):
    ccw = (R.net_rotation(RES[(name, MAIN)]["planes"][0]) > 0).astype(float)
    x = CURV[(name, MAIN[0])][0] / 100.0
    if ccw.min() == ccw.max():
        print(f"{name}: logistic fit not possible, all conditions rotate the same way")
        continue
    try:
        fit = sm.Logit(ccw, sm.add_constant(x)).fit(disp=0)
        print(f"{name}: logistic fit of CCW on curvature (per 100 deg): slope {fit.params[1]:.3f}, p = {fit.pvalues[1]:.3f}, n = {len(x)}, {int(ccw.sum())} CCW")
    except Exception as e:
        print(f"{name}: logistic fit failed ({type(e).__name__}: {e})")

**Result: partially reproduced, resting on the assumption about the onset column.** Under the main reading (offline onset) 88.2% of the 34 clockwise-curving J-array conditions rotate counter-clockwise, below 90% and above 75%; the straight and counter-clockwise classes of J-array and all three classes of N-array are at 98.1% or more. Under the two move_begins readings, with the hand paths also aligned on move_begins_time, every class is at 92.5% or more (J-array clockwise class 93.8% of 32), which would make the claim reproduced. The velocity-heading variant gives the same split by reading. Four of the five J-array conditions that do not rotate counter-clockwise curve clockwise, by 107 to 148 degrees, against a median of -106 degrees for the clockwise class; the fifth curves counter-clockwise (33 degrees). In J-array the logistic fit finds that the more a reach curves clockwise, the less likely its state rotates counter-clockwise (slope 1.524 per 100 degrees, p = 0.029); in N-array it finds no relation (slope -0.175, p = 0.873).


## fig3-phase-from-prep-state

> The rotation phase follows from the preparatory state; knowing the initial state in the jPC plane, the subsequent states are reasonably predictable.

Criterion: writing each condition's state in the plane as a complex number, the state at +150 ms is predicted from the state at -50 ms by one complex gain shared by all conditions, fitted leaving one condition out. Reproduced when the cross-validated R2 is at least 0.5 in each dataset and above the median R2 of shuffle 3 (movement activity reassigned); partially reproduced when this holds in one dataset.

The shuffle-3 runs here are this claim's own (20 runs from its seed), with the same dividing point as suppfig3-shuffle-controls: the median go-cue time relative to movement onset plus 50 ms (Supplementary Figure 2 legend). The R2 of a shuffle is that of its own jPCA plane.

The dividing point (-277 and -268 ms) lies before the analysed window, so under shuffle 3 the states at -50 and +150 ms both carry the same reassigned movement pattern, and the state at -50 ms predicts the one at +150 ms by construction; the comparison may then not test whether the preparatory state predicts the phase. Two variants are shown, which do not decide the outcome: the shuffle-3 runs projected onto the original data's plane (R2 from -50 ms), and the state at +150 ms predicted from the state at the dividing point, for the data and for shuffle 3, both in the original plane. Whether the criterion's reference should be changed is asked of a reader (`questions/churchland-2012.yaml`).

In [ ]:
def phase_r2(Z):
    z = Z[0] + 1j * Z[1]
    a, b = z[:, 0], z[:, -1]
    pred = np.empty_like(b)
    for c in range(len(a)):
        m = np.arange(len(a)) != c
        g = np.sum(np.conj(a[m]) * b[m]) / np.sum(np.abs(a[m]) ** 2)
        pred[c] = g * a[c]
    return 1 - np.sum(np.abs(b - pred) ** 2) / np.sum(np.abs(b - b.mean()) ** 2)

def dividing_ms(tr, col):
    return int(round(np.median((tr.go_cue_time - tr[col]).values) * 1000 + 50))

PH, PHV = {}, {}
for name in ("J-array", "N-array"):
    D = DATA[name]
    for rd in READINGS:
        div = dividing_ms(D["trials"], rd[0])
        r = D["rates"][rd[0]]
        o = RES[(name, rd)]
        orig = phase_r2(o["planes"][0])
        orig_div = phase_r2(R.project(R.preprocess(r, rd[1], out_times=[div, 150]), o))
        offs = SEED_OFFSETS if rd == MAIN else [0]
        meds, var_own, var_div = [], [], []
        for off in offs:
            rng = np.random.default_rng(seed("fig3-phase-from-prep-state", off))
            vals = []
            for _ in range(20):
                rs = R.shuffle_reassign(r, div, rng.permutation(108))
                Xs = R.preprocess(rs, rd[1])
                vals.append(phase_r2(R.jpca(Xs, 6)["planes"][0]))
                if off == 0:
                    var_own.append(phase_r2(R.project(Xs, o)))
                    var_div.append(phase_r2(R.project(R.preprocess(rs, rd[1], out_times=[div, 150]), o)))
            meds.append(np.median(vals))
        PH[(name, rd)] = (orig, meds)
        PHV[(name, rd)] = (orig_div, np.median(var_own), np.median(var_div))
        print(f"{name} | {rlabel(rd)}: dividing point {div} ms; cross-validated R2 {orig:.3f}; shuffle-3 median R2 {meds[0]:.3f}"
              + (f" (five seeds: {', '.join(f'{m:.3f}' for m in meds)})" if len(meds) > 1 else ""))
        print(f"   variants in the original plane: shuffle-3 median R2 from -50 ms {np.median(var_own):.3f}; "
              f"+150 ms predicted from the state at the dividing point: data {orig_div:.3f}, shuffle-3 median {np.median(var_div):.3f}")
for rd in READINGS:
    ok = [PH[(n, rd)][0] >= 0.5 and PH[(n, rd)][0] > PH[(n, rd)][1][0] for n in ("J-array", "N-array")]
    print(rlabel(rd), "->", "reproduced" if all(ok) else ("partially_reproduced" if any(ok) else "not_reproduced"))
for rd in READINGS:
    ok = [PHV[(n, rd)][0] >= 0.5 and PHV[(n, rd)][0] > PHV[(n, rd)][2] for n in ("J-array", "N-array")]
    print(rlabel(rd), "| variant, prediction from the dividing point in the original plane ->", "reproduced" if all(ok) else ("partially_reproduced" if any(ok) else "not_reproduced"))
for rd in READINGS:
    ok = [PH[(n, rd)][0] >= 0.5 and PH[(n, rd)][0] > PHV[(n, rd)][1] for n in ("J-array", "N-array")]
    print(rlabel(rd), "| variant, shuffle-3 runs projected onto the original plane, from -50 ms ->", "reproduced" if all(ok) else ("partially_reproduced" if any(ok) else "not_reproduced"))


**Result: partially reproduced.** The cross-validated R2 is 0.568 in J-array and 0.676 in N-array, both at least 0.5. In N-array it is above the shuffle-3 median (0.600); in J-array it is below it (0.686), so the criterion holds in one dataset. All four readings and the five seeds give the same outcome (J-array shuffle-3 medians 0.678 to 0.688 over the seeds). In the original plane the shuffle-3 runs predict less well than the data in both datasets (medians 0.429 in J-array and 0.611 in N-array, from -50 ms), which would make the claim reproduced. Predicted from the state at the dividing point, the data reach only 0.103 (J-array) and 0.252 (N-array), against shuffle-3 medians of 0.055 and 0.058: the data predict better than the shuffles, but well below 0.5.


## var-captured-jpc-plane

> The first jPCA plane captures an average of 28% of the total data variance (typically 20 to 35%).

Criterion: (a) the mean of J-array and N-array within 28 +- 2.8%, (b) each of the two within 18 to 38.5%. Reproduced when (a) or (b) holds.

In [ ]:
for rd in READINGS + ["smooth24"]:
    v = [100 * RES[(n, rd)]["plane_var"][0] for n in ("J-array", "N-array")]
    a = abs(np.mean(v) - 28) <= 2.8
    b = all(18 <= x <= 38.5 for x in v)
    lab = rlabel(rd) if rd != "smooth24" else "24 ms Gaussian (variant)"
    print(f"{lab}: J-array {v[0]:.1f}%, N-array {v[1]:.1f}%, mean {np.mean(v):.1f}% | (a) {a} | (b) {b} -> {'reproduced' if (a or b) else 'not_reproduced'}")

**Result: not reproduced, resting on the assumption about the onset column.** Under the main reading the first plane captures 14.9% (J-array) and 22.9% (N-array), mean 18.9%: the mean is outside 28 +- 2.8% and J-array is below 18%. Under the move_begins readings J-array captures 18.7 to 19.0% and N-array 21.1 to 22.0%, all inside 18 to 38.5%, which would make it reproduced under reading (b). The 24 ms variant gives 15.6% and 23.3%. The range of reading (b), "typically 20-35%", comes from the Supplementary Figure 5 legend, which is about the per-area analysis; whether it applies to the whole population is asked of a reader. Under the main reading the outcome does not depend on it.


## pc23-variance

> The plane of PC2 and PC3 captures 29% of the total variance on average, about as much as the first jPCA plane, so the jPCA plane captures patterns already present in the top PCs.

Criterion: readings (a) the mean of J-array and N-array within 29 +- 2.9%, (b) each dataset within it. Reproduced when (a) or (b) holds and, in each dataset, the jPCA plane captures at least 80% as much variance as PC2 and PC3 together; partially reproduced when only one of these two parts holds.

In [ ]:
for rd in READINGS + ["smooth24"]:
    v = [100 * RES[(n, rd)]["pc_var"][1:3].sum() for n in ("J-array", "N-array")]
    j = [100 * RES[(n, rd)]["plane_var"][0] for n in ("J-array", "N-array")]
    p1 = abs(np.mean(v) - 29) <= 2.9 or all(abs(x - 29) <= 2.9 for x in v)
    p2 = all(jj >= 0.8 * vv for jj, vv in zip(j, v))
    lab = rlabel(rd) if rd != "smooth24" else "24 ms Gaussian (variant)"
    print(f"{lab}: PC2+PC3 J {v[0]:.1f}%, N {v[1]:.1f}%, mean {np.mean(v):.1f}% | jPCA/PC2+3 J {j[0] / v[0]:.2f}, N {j[1] / v[1]:.2f} | "
          f"values {p1}, ratio {p2} -> {'reproduced' if p1 and p2 else ('partially_reproduced' if p1 or p2 else 'not_reproduced')}")

**Result: not reproduced, resting on the assumption about the onset column.** Under the main reading PC2 and PC3 capture 19.2% (J-array) and 28.3% (N-array), mean 23.8%: the mean and the J-array value are outside 29 +- 2.9%. The jPCA plane captures 0.78 (J-array) and 0.81 (N-array) as much variance as PC2 and PC3, so the 80% part fails in J-array. Under the move_begins readings the ratio part holds (0.84 to 0.95) and the values part does not, which would make it partially reproduced.

## suppfig4-j-array-planes and suppfig4-planes-range

> suppfig4-j-array-planes: in J-array the first three jPCA planes found within the top 10 PCs capture 22%, 19% and 12% of the variance (54% together), against 64% for the top six PCs, and each plane holds rotations coherent across conditions.

> suppfig4-planes-range: every dataset contains two, and usually three, planes with rotations, together capturing 50 to 70% of the variance.

Criteria: J-array, reproduced when all five values are within tolerance (22 +- 2.2, 19 +- 1.9, 12 +- 1.2, 54 +- 5.4, 64 +- 6.4) and at least 75% of the conditions rotate in the plane's direction in each of the three planes; partially when the three-plane total and the top-six-PC value are within tolerance. Range: reproduced when in each dataset the first three planes together capture 45 to 77% and at least the first two planes have 75% or more of the conditions rotating in the plane's direction; partially when this holds in one dataset.

The planes are ordered by the magnitude of their M_skew eigenvalues (Methods: "the largest two (complex-conjugate) imaginary eigenvalues" define the first plane); each plane is oriented as the first.

In [ ]:
R10 = {}
for name in ("J-array", "N-array"):
    for rd in READINGS:
        R10[(name, rd)] = R.jpca(R.preprocess(DATA[name]["rates"][rd[0]], rd[1]), 10, 3)
def s4(o):
    pv = 100 * o["plane_var"]
    fr = [(R.net_rotation(z) > 0).mean() for z in o["planes"]]
    hz = [R.freq_hz(e)[1] for e in o["plane_eig"]]
    return pv, pv.sum(), 100 * o["pc_var_all"][:6].sum(), fr, hz
for rd in READINGS:
    pv, tot, top6, fr, hz = s4(R10[("J-array", rd)])
    exp = [(22, 2.2), (19, 1.9), (12, 1.2)]
    within = [abs(p - e) <= t for p, (e, t) in zip(pv, exp)] + [abs(tot - 54) <= 5.4, abs(top6 - 64) <= 6.4]
    out = "reproduced" if all(within) and all(f >= 0.75 for f in fr) else ("partially_reproduced" if within[3] and within[4] else "not_reproduced")
    print(f"J-array | {rlabel(rd)}: planes {', '.join(f'{p:.1f}%' for p in pv)} (total {tot:.1f}%), top six PCs {top6:.1f}%, "
          f"CCW {', '.join(f'{100 * f:.0f}%' for f in fr)}, frequencies {', '.join(f'{h:.2f}' for h in hz)} Hz | within {within} -> {out}")

**Result: not reproduced.** In J-array the three planes capture 13.3%, 13.3% and 12.2% (38.8% together) and the top six PCs 51.9%, against 22, 19, 12, 54 and 64%; only the third plane is within tolerance. The rotations are coherent in each plane (98%, 97% and 92% of conditions counter-clockwise). The four readings agree.

In [ ]:
for rd in READINGS:
    ok = []
    for name in ("J-array", "N-array"):
        pv, tot, top6, fr, hz = s4(R10[(name, rd)])
        good = 45 <= tot <= 77 and fr[0] >= 0.75 and fr[1] >= 0.75
        ok.append(good)
        print(f"{name} | {rlabel(rd)}: three planes {tot:.1f}% ({', '.join(f'{p:.1f}' for p in pv)}), CCW {', '.join(f'{100 * f:.0f}%' for f in fr)} -> {good}")
    print("   ->", "reproduced" if all(ok) else ("partially_reproduced" if any(ok) else "not_reproduced"))

**Result: partially reproduced.** In N-array the three planes capture 47.7%, inside 45 to 77%, with 97% and 100% of conditions rotating in the first two planes; in J-array they capture 38.8%, below 45%, although 98% and 97% rotate. The four readings agree (J-array 37.7 to 38.8%, N-array 45.8 to 48.8%).

## fig6a-angle-peak

> The distribution of the angle between the neural state and its derivative in the first jPCA plane, over all conditions and times, peaks near pi/2 (rotation), not near 0.

Criterion: peak of a 36-bin histogram over -pi to pi. Reproduced when the peak lies between 0.3 pi and 0.7 pi in each dataset; partially reproduced in one.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for i, name in enumerate(("J-array", "N-array")):
    for rd in READINGS:
        ang = R.state_derivative_angles(RES[(name, rd)]["planes"][0])
        pk, h, e = R.angle_peak(ang)
        print(f"{name} | {rlabel(rd)}: peak at {pk / np.pi:.3f} pi ({h.max()} of {h.sum()} angles in the peak bin); "
              f"share of angles within 0.25 pi of pi/2: {np.mean(np.abs(ang - np.pi / 2) < np.pi / 4):.2f}")
        if rd == MAIN:
            ax[i].bar((e[:-1] + e[1:]) / 2 / np.pi, h, width=2 / 36); ax[i].set_title(name); ax[i].set_xlabel("angle / pi")
plt.show()
for rd in READINGS:
    ok = [0.3 <= R.angle_peak(R.state_derivative_angles(RES[(n, rd)]["planes"][0]))[0] / np.pi <= 0.7 for n in ("J-array", "N-array")]
    print(rlabel(rd), "->", "reproduced" if all(ok) else ("partially_reproduced" if any(ok) else "not_reproduced"))

**Result: reproduced.** The histogram peaks at 0.528 pi in J-array and 0.639 pi in N-array under the main reading, and at 0.528 to 0.583 pi and 0.639 pi under all four readings, between 0.3 pi and 0.7 pi.

## fig6b-skew-fit

> For the neural data the rotational (skew-symmetric) dynamics matrix fits the data nearly as well as the best unconstrained linear dynamics matrix, both in the 6-D space and in the first jPCA plane.

Criterion: the ratio R2(M_skew) / R2(M). Reproduced when the 6-D ratio is at least 0.7 and the plane ratio at least 0.75 in each dataset; partially when both hold in one dataset, or one of the two holds in both. In the plane, M and M_skew are 2 x 2 matrices fitted to the projection onto the first jPCA plane (Figure 6b legend: "the rank 2 matrices that capture dynamics in the first jPCA plane").

In [ ]:
for rd in READINGS + ["smooth24"]:
    res = []
    for name in ("J-array", "N-array"):
        s = summarise(RES[(name, rd)])
        res.append((s["ratio_6d"] >= 0.7, s["ratio_plane"] >= 0.75))
        lab = rlabel(rd) if rd != "smooth24" else "24 ms Gaussian (variant)"
        print(f"{name} | {lab}: 6-D R2 M {s['R2_M']:.3f}, M_skew {s['R2_skew']:.3f}, ratio {s['ratio_6d']:.3f}; "
              f"plane R2 M {s['R2_M_plane']:.3f}, M_skew {s['R2_skew_plane']:.3f}, ratio {s['ratio_plane']:.3f}")
    both = [a and b for a, b in res]
    out = "reproduced" if all(both) else ("partially_reproduced" if any(both) or all(a for a, b in res) or all(b for a, b in res) else "not_reproduced")
    print("   ->", out)

**Result: reproduced.** The 6-D ratio R2(M_skew)/R2(M) is 0.918 (J-array) and 0.909 (N-array), and the plane ratio 0.946 and 0.953. Under all four readings and the 24 ms variant the 6-D ratios are 0.870 or more and the plane ratios 0.908 or more.

## rotation-frequency

> The rotations are brief and fast, up to about 2.5 Hz; the largest eigenvalues of M_skew in the monkey A and B datasets are 8.8 to 13.5 radians/s.

Criterion: frequency is the magnitude of the largest eigenvalue of M_skew (radians/s) over 2 pi. Reproduced when it lies within 1.26 to 2.75 Hz in each dataset; partially when in one. The derivative is the difference of samples 10 ms apart, so an eigenvalue per sample is multiplied by 100 to give radians/s.

In [ ]:
for rd in READINGS + ["smooth24"]:
    ok = []
    for name in ("J-array", "N-array"):
        s = summarise(RES[(name, rd)])
        ok.append(1.26 <= s["freq_hz"] <= 2.75)
        lab = rlabel(rd) if rd != "smooth24" else "24 ms Gaussian (variant)"
        print(f"{name} | {lab}: largest |eigenvalue| {s['max_eig_rad_s']:.2f} rad/s = {s['freq_hz']:.3f} Hz")
    print("   ->", "reproduced" if all(ok) else ("partially_reproduced" if any(ok) else "not_reproduced"))

**Result: reproduced, resting on the assumption about the onset column.** The largest eigenvalue of M_skew is 8.44 rad/s (1.344 Hz) in J-array and 8.81 rad/s (1.402 Hz) in N-array, inside 1.26 to 2.75 Hz. Under the move_begins readings J-array gives 1.063 to 1.098 Hz, below the range, which would make it partially reproduced. The 24 ms variant gives 1.290 and 1.372 Hz.

## suppfig3-shuffle-controls

> Three shuffle controls that keep preparatory activity and invert or reassign the peri-movement activity largely remove the rotational structure found by jPCA, so jPCA does not find rotations that are not there.

Criterion: the dividing point is the median go-cue time relative to movement onset plus 50 ms. Shuffle 1 (half the conditions inverted per unit) and 3 (movement activity reassigned) are run 20 times with fixed seeds, shuffle 2 once. Rotation strength is R2(M_skew) in 6-D. Reproduced when, in each dataset and for each shuffle, every run is below the original and the median is at most 75% of it; partially when this holds for at least two shuffles in each dataset or all three in one. The fraction of conditions rotating counter-clockwise and the R2 ratio are shown as well. The Supplementary Figure 2 legend says shuffles 2 and 3 are "not expected to remove all rotational structure (most such structure is merely sign-inverted)"; whether R2(M_skew) is then the measure the paper's claim rests on is asked of a reader (`questions/churchland-2012.yaml`). The claim is tested as written.

Shuffle 1 inverts exactly half of the conditions (54 of 108), drawn per unit (Supplementary Figure 2 legend: "inverted for half of the conditions, selected at random ... performed separately for each neuron"). Inversion reflects the rate after the dividing point about its value there; reassignment adds the other condition's change from its dividing-point rate to this condition's rate at that point.

In [ ]:
def shuffle_runs(name, rd, off):
    D = DATA[name]
    r = D["rates"][rd[0]]
    div = dividing_ms(D["trials"], rd[0])
    rng = np.random.default_rng(seed("suppfig3-shuffle-controls", off))
    nu = r.shape[0]
    out = {1: [], 2: [], 3: []}
    for _ in range(20):
        mask = np.zeros((nu, 108), bool)
        for u in range(nu):
            mask[u, rng.permutation(108)[:54]] = True
        out[1].append(summarise(R.jpca(R.preprocess(R.shuffle_invert(r, div, mask), rd[1]), 6)))
    out[2].append(summarise(R.jpca(R.preprocess(R.shuffle_invert(r, div, np.ones((nu, 108), bool)), rd[1]), 6)))
    for _ in range(20):
        out[3].append(summarise(R.jpca(R.preprocess(R.shuffle_reassign(r, div, rng.permutation(108)), rd[1]), 6)))
    return div, out

def shuffle_verdict(orig, out):
    res = {}
    for k, runs in out.items():
        v = np.array([s["R2_skew"] for s in runs])
        res[k] = bool(np.all(v < orig) and np.median(v) <= 0.75 * orig)
    return res

SH = {}
for name in ("J-array", "N-array"):
    for rd in READINGS:
        offs = SEED_OFFSETS if rd == MAIN else [0]
        for off in offs:
            SH[(name, rd, off)] = shuffle_runs(name, rd, off)
for name in ("J-array", "N-array"):
    for rd in READINGS:
        orig = summarise(RES[(name, rd)])
        div, out = SH[(name, rd, 0)]
        print(f"{name} | {rlabel(rd)} | dividing point {div} ms | original R2_skew {orig['R2_skew']:.3f}, CCW {100 * orig['ccw_frac']:.0f}%, ratio {orig['ratio_6d']:.3f}")
        for k, runs in out.items():
            v = np.array([s["R2_skew"] for s in runs])
            print(f"   shuffle {k} ({len(v)} runs): R2_skew median {np.median(v):.3f} ({np.median(v) / orig['R2_skew']:.2f} of original), range {v.min():.3f} to {v.max():.3f}, "
                  f"runs at or above original {int((v >= orig['R2_skew']).sum())}; CCW median {100 * np.median([s['ccw_frac'] for s in runs]):.0f}%; ratio median {np.median([s['ratio_6d'] for s in runs]):.3f}")
        print("   meets the criterion per shuffle:", shuffle_verdict(orig["R2_skew"], out))
print("Seed sensitivity, main reading: median R2_skew of shuffles 1 and 3 as a fraction of the original, per seed offset")
for name in ("J-array", "N-array"):
    orig = RES[(name, MAIN)]["R2_skew"]
    for off in SEED_OFFSETS:
        div, out = SH[(name, MAIN, off)]
        print(f"   {name} +{off}: shuffle 1 {np.median([s['R2_skew'] for s in out[1]]) / orig:.3f}, shuffle 3 {np.median([s['R2_skew'] for s in out[3]]) / orig:.3f}, verdict {shuffle_verdict(orig, out)}")
def sh_outcome(rd, off=0):
    per = [shuffle_verdict(RES[(n, rd)]["R2_skew"], SH[(n, rd, off)][1]) for n in ("J-array", "N-array")]
    if all(all(p.values()) for p in per): return "reproduced"
    if all(sum(p.values()) >= 2 for p in per) or any(all(p.values()) for p in per): return "partially_reproduced"
    return "not_reproduced"
for rd in READINGS:
    print(rlabel(rd), "->", sh_outcome(rd))
print("main reading under the five seeds ->", [sh_outcome(MAIN, off) for off in SEED_OFFSETS])

**Result: partially reproduced, resting on the assumption about the onset column.** Under the main reading, in J-array every run of every shuffle is below the original R2(M_skew) of 0.381, and the medians are 0.01 (shuffle 1), 0.63 (shuffle 2) and 0.75 (shuffle 3) of it, so all three shuffles meet the criterion. In N-array shuffle 1 removes the structure (0.01 of the original of 0.527) but shuffles 2 and 3 leave 0.76 and 0.87 of it. The criterion's partial outcome (all three shuffles in one dataset) holds under each of the five seeds (J-array shuffle-3 medians 0.725 to 0.745 of the original). The other range window also gives partially reproduced (two shuffles in each dataset); the two move_begins readings give not reproduced, as J-array shuffle 3 is then above 0.75 and N-array meets only shuffle 1. After shuffles 2 and 3 the median share of conditions rotating counter-clockwise stays at 93 to 98%.


## suppfig5-areas-variance and suppfig5-m1-faster

> suppfig5-areas-variance: analysed separately, the PMd and M1 arrays each show clear rotations; the jPCA plane captures 29% (PMd) and 29% (M1) in J-array and 32% and 28% in N-array.

> suppfig5-m1-faster: the angular velocity of the rotations is slightly higher for M1 than for PMd in each dataset.

Criteria: units split by the electrode group of their array (PMd group 1, M1 group 2). Areas: reproduced when all four values are within tolerance (29 +- 2.9, 29 +- 2.9, 32 +- 3.2, 28 +- 2.8) and at least 75% of conditions rotate counter-clockwise in each; partially when all four lie in 18 to 38.5% with 75% rotating. Faster: frequency of the first jPCA plane per array; reproduced when M1 is higher than PMd in both datasets; partially in one. The sub-Nitschke ses-20100910 session is shown against the N-9-10-2010 values (26% and 24%) as a variant.

In [ ]:
AREA = {}
for name in ("J-array", "N-array", "N-9-10-2010"):
    D = DATA[name]
    rds = READINGS if name != "N-9-10-2010" else [MAIN]
    for rd in rds:
        for g, area in (("1", "PMd"), ("2", "M1")):
            m = D["groups"] == g
            o = R.jpca(R.preprocess(D["rates"][rd[0]][m], rd[1]), 6)
            AREA[(name, rd, area)] = summarise(o)
expv = {("J-array", "PMd"): (29, 2.9), ("J-array", "M1"): (29, 2.9), ("N-array", "PMd"): (32, 3.2), ("N-array", "M1"): (28, 2.8),
        ("N-9-10-2010", "PMd"): (26, 2.6), ("N-9-10-2010", "M1"): (24, 2.4)}
for rd in READINGS:
    w, rng_ok = [], []
    for name in ("J-array", "N-array"):
        for area in ("PMd", "M1"):
            s = AREA[(name, rd, area)]
            e, t = expv[(name, area)]
            w.append(abs(s["plane_var"] - e) <= t and s["ccw_frac"] >= 0.75)
            rng_ok.append(18 <= s["plane_var"] <= 38.5 and s["ccw_frac"] >= 0.75)
            print(f"{name} {area} | {rlabel(rd)}: plane variance {s['plane_var']:.1f}% (paper {e}%), CCW {100 * s['ccw_frac']:.0f}%, frequency {s['freq_hz']:.3f} Hz")
    print("   ->", "reproduced" if all(w) else ("partially_reproduced" if all(rng_ok) else "not_reproduced"))
for area in ("PMd", "M1"):
    s = AREA[("N-9-10-2010", MAIN, area)]
    print(f"variant N-9-10-2010 {area}: plane variance {s['plane_var']:.1f}% (paper {expv[('N-9-10-2010', area)][0]}%), CCW {100 * s['ccw_frac']:.0f}%, frequency {s['freq_hz']:.3f} Hz")

**Result: not reproduced.** Split by array, the first plane captures 21.6% (PMd) and 20.2% (M1) in J-array and 17.7% and 27.1% in N-array, against 29, 29, 32 and 28%; only N-array M1 is within tolerance, and N-array PMd is below 18%. Each array rotates (91 to 99% of conditions counter-clockwise under the main reading). The four readings agree (N-array PMd 17.6 to 17.9%). The N-9-10-2010 variant gives 14.7% (PMd) and 20.8% (M1) against 26% and 24%.

In [ ]:
for rd in READINGS:
    ok = []
    for name in ("J-array", "N-array"):
        p, m = AREA[(name, rd, "PMd")]["freq_hz"], AREA[(name, rd, "M1")]["freq_hz"]
        ok.append(m > p)
        print(f"{name} | {rlabel(rd)}: PMd {p:.3f} Hz, M1 {m:.3f} Hz, M1 - PMd {m - p:+.3f} Hz")
    print("   ->", "reproduced" if all(ok) else ("partially_reproduced" if any(ok) else "not_reproduced"))
p, m = AREA[("N-9-10-2010", MAIN, "PMd")]["freq_hz"], AREA[("N-9-10-2010", MAIN, "M1")]["freq_hz"]
print(f"variant N-9-10-2010: PMd {p:.3f} Hz, M1 {m:.3f} Hz, M1 - PMd {m - p:+.3f} Hz")

**Result: reproduced.** The first-plane frequency is higher for M1 than for PMd in J-array (2.020 against 0.844 Hz) and in N-array (1.488 against 0.973 Hz) under the main reading, and in both datasets under all four readings; the N-9-10-2010 variant agrees (1.659 against 1.063 Hz). M1 is 2.39 (J-array) and 1.53 (N-array) times faster than PMd under the main reading, and 1.41 to 2.39 times under the four readings.

## fig6-velocity-model and fig6-complex-kinematic-model

> fig6-velocity-model: a simulated population of units tuned to hand velocity and speed, with preparatory activity tuned to reach endpoint, matched to the recorded data, shows no robust or consistent rotations; M_skew fits it poorly and its angle distribution peaks slightly above zero.

> fig6-complex-kinematic-model: a simulated population tuned to hand position, velocity, acceleration and jerk (with the Methods' sensitivities) produces multiphasic responses but no consistent rotations; M_skew fits it poorly.

Criterion: one simulated dataset per array dataset from that session's condition-mean hand kinematics, analysed as the data. Reproduced when, in each dataset, its 6-D R2 ratio is below 0.6 and below the neural ratio and its angle peak is below pi/4; partially when this holds in one dataset. The angle peak is read as a peak near zero, |peak| < pi/4: a peak near -pi, where the state and its derivative point in opposite directions, is not "slightly above zero". The stricter reading 0 <= peak < pi/4 is shown as well.

The Methods describe the factors and the sensitivities but not how far the simulated rates lead the hand, nor the time course of the preparatory term. The main reading uses a lead of 100 ms (the usual lead of motor cortical activity in velocity-tuning models such as ref. 27) and a preparatory term that falls to zero over 100 ms centred on the time the movement term starts, 100 ms before hand onset (`prep="lead"`; `reach.simulate_population`, docstring): in the model both terms are neural, so preparatory activity gives way when movement activity begins, before the analysed window. The other readings are the same fall centred on hand onset (`switch`, which with a 100 ms lead puts the change from preparatory to movement tuning inside the analysed window), a lead of 0 ms (where `lead` and `switch` coincide), and a preparatory term that persists. Every reading is run under the five seeds. Each simulated unit is matched to one recorded unit's mean rate, range and trial counts, and spikes are drawn from a gamma-interval process of order 2 (Methods). Simulations are run under the main pipeline reading (offline onset, range over the analysed window).


In [ ]:
SIM_READINGS = [(100, "lead"), (100, "switch"), (0, "switch"), (100, "persist"), (0, "persist")]
SIM_ANG = {}
SIM = {}
t_sim = time.time()
for model, claim in (("velocity", "fig6-velocity-model"), ("complex", "fig6-complex-kinematic-model")):
    for name in ("J-array", "N-array"):
        D = DATA[name]
        kin = R.kinematics(D["hand_mean"], TH)
        rates = D["rates"]["offline_move_onset_time"]
        for sr in SIM_READINGS:
            for off in SEED_OFFSETS:
                rng = np.random.default_rng(seed(claim, off))
                S = R.simulate_population(model, kin, TH, rates, D["n"], rng, lag_ms=sr[0], prep=sr[1])
                sim_rates = R.smooth_rates(S, D["n"])
                o = R.jpca(R.preprocess(sim_rates), 6)
                s = summarise(o)
                ang = R.state_derivative_angles(o["planes"][0])
                s["near0"] = float(np.mean(np.abs(ang) < np.pi / 4))
                s["near_half_pi"] = float(np.mean(np.abs(ang - np.pi / 2) < np.pi / 4))
                SIM[(model, name, sr, off)] = s
                if off == 0:
                    SIM_ANG[(model, name, sr)] = ang
print(f"simulations took {time.time() - t_sim:.0f} s")

def sim_ok(model, name, sr, off=0):
    s = SIM[(model, name, sr, off)]
    neural = summarise(RES[(name, MAIN)])["ratio_6d"]
    # "angle peak below pi/4": a peak near zero, |peak| < pi/4; a peak near -pi
    # (state and derivative opposed) does not count. 0 <= peak < pi/4 is shown too.
    return s["ratio_6d"] < 0.6 and s["ratio_6d"] < neural and abs(s["peak_over_pi"]) < 0.25

def sim_ok_nonneg(model, name, sr, off=0):
    s = SIM[(model, name, sr, off)]
    neural = summarise(RES[(name, MAIN)])["ratio_6d"]
    return s["ratio_6d"] < 0.6 and s["ratio_6d"] < neural and 0 <= s["peak_over_pi"] < 0.25

def sim_outcome(model, sr, off=0, test=None):
    test = sim_ok if test is None else test
    ok = [test(model, n, sr, off) for n in ("J-array", "N-array")]
    return "reproduced" if all(ok) else ("partially_reproduced" if any(ok) else "not_reproduced")

def show_sim(model):
    for sr in SIM_READINGS:
        for name in ("J-array", "N-array"):
            s = SIM[(model, name, sr, 0)]
            neural = summarise(RES[(name, MAIN)])
            print(f"{model} | lead {sr[0]} ms, preparatory {sr[1]} | {name}: 6-D ratio {s['ratio_6d']:.3f} (neural {neural['ratio_6d']:.3f}), "
                  f"plane ratio {s['ratio_plane']:.3f}, angle peak {s['peak_over_pi']:.3f} pi, CCW {100 * s['ccw_frac']:.0f}%, plane variance {s['plane_var']:.1f}%, "
                  f"angles within pi/4 of 0 {s['near0']:.2f} and of pi/2 {s['near_half_pi']:.2f} -> {sim_ok(model, name, sr)}")
        print("   ->", sim_outcome(model, sr), "| with 0 <= peak < pi/4:", sim_outcome(model, sr, test=sim_ok_nonneg))
        for name in ("J-array", "N-array"):
            vals = [SIM[(model, name, sr, o)] for o in SEED_OFFSETS]
            peaks = ", ".join(f"{v['peak_over_pi']:.3f}" for v in vals)
            print(f"   five seeds, {name}: 6-D ratio {min(v['ratio_6d'] for v in vals):.3f} to {max(v['ratio_6d'] for v in vals):.3f}, "
                  f"angle peaks {peaks} pi, angles within pi/4 of 0 {min(v['near0'] for v in vals):.2f} to {max(v['near0'] for v in vals):.2f}")
        print("   five seeds ->", [sim_outcome(model, sr, o) for o in SEED_OFFSETS])
    fig, ax = plt.subplots(1, 2, figsize=(9, 3))
    for i, name in enumerate(("J-array", "N-array")):
        h, e = np.histogram(SIM_ANG[(model, name, SIM_READINGS[0])], bins=36, range=(-np.pi, np.pi))
        hn, _ = np.histogram(R.state_derivative_angles(RES[(name, MAIN)]["planes"][0]), bins=36, range=(-np.pi, np.pi))
        c = (e[:-1] + e[1:]) / 2 / np.pi
        ax[i].bar(c, h, width=2 / 36, alpha=0.7, label=f"{model} model")
        ax[i].step(c, hn, where="mid", color="k", label="neural")
        ax[i].set_title(f"{name}, main reading"); ax[i].set_xlabel("angle / pi"); ax[i].legend(fontsize=7)
        print(f"{name}: histogram counts, {model} model {h.min()} to {h.max()}, neural {hn.min()} to {hn.max()} (all bins shown)")
    plt.show()
show_sim("velocity")

**Result: partially reproduced, resting on the assumptions about the simulated model.** Under the main reading (100 ms lead, preparatory term falling as the movement term starts) M_skew fits the simulated population poorly in both datasets (6-D ratio 0.590 in J-array and 0.153 in N-array, below 0.6 and below the neural 0.918 and 0.909). The N-array angle distribution peaks at 0.083 pi, below pi/4, and the J-array one at 0.472 pi, above it, so the criterion holds in N-array only. The five seeds give the same outcome (J-array 6-D ratios 0.517 to 0.590 and peaks 0.472 to 0.528 pi; N-array peaks 0.083 to 0.139 pi). The readings differ: with the fall centred on hand onset (100 ms lead) or a persisting preparatory term with a 100 ms lead the outcome is not reproduced (peaks 0.417 and 0.639 pi, and 0.639 and 0.417 pi; with the persisting term one seed of five gives partially reproduced), as it is with a persisting term and no lead (peaks 0.250 and 0.361 pi); with no lead and the falling term it is reproduced (peaks -0.083 and 0.083 pi). In every reading and seed the 6-D ratio is below the neural ratio (0.056 to 0.590). The share of simulated angles within pi/4 of zero is 0.28 (J-array) and 0.39 (N-array) under the main reading, against 0.36 and 0.36 within pi/4 of pi/2.


In [ ]:
show_sim("complex")

**Result: not reproduced, resting on the assumptions about the simulated model.** Under the main reading M_skew fits poorly in both datasets (6-D ratio 0.074 in J-array and 0.060 in N-array, against the neural 0.918 and 0.909), but neither angle distribution peaks near zero: the peaks are at -0.972 pi (J-array) and 0.806 pi (N-array), and only 0.11 and 0.10 of the angles lie within pi/4 of zero. The five seeds all give not reproduced (J-array peaks between 0.528 and 0.972 pi in magnitude, N-array between 0.750 and 0.972 pi). Under the claim's seed the other readings give not reproduced, except no lead with the falling preparatory term, which gives partially reproduced under every seed (J-array peaks 0.028 to 0.139 pi; N-array 0.250 to 0.306 pi, at or above pi/4). No lead with a persisting term gives not reproduced under the claim's seed and partially reproduced under three of the five seeds (J-array peaks -0.917, -0.917, 0.083, 0.139 and 0.139 pi). The part of the statement that M_skew fits poorly holds under every reading and seed (6-D ratios 0.060 to 0.398).


## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| array-dataset-composition | partially reproduced |
| fig3ef-same-direction | reproduced |
| fig3-independent-of-curvature | partially reproduced (assumed onset column) |
| fig3-phase-from-prep-state | partially reproduced |
| var-captured-jpc-plane | not reproduced (assumed onset column) |
| pc23-variance | not reproduced (assumed onset column) |
| suppfig4-j-array-planes | not reproduced |
| suppfig4-planes-range | partially reproduced |
| fig6a-angle-peak | reproduced |
| fig6b-skew-fit | reproduced |
| rotation-frequency | reproduced (assumed onset column) |
| suppfig3-shuffle-controls | partially reproduced (assumed onset column) |
| suppfig5-areas-variance | not reproduced |
| suppfig5-m1-faster | reproduced |
| fig6-velocity-model | partially reproduced (assumed model details) |
| fig6-complex-kinematic-model | not reproduced (assumed model details) |
| fig1-leech-and-walking | blocked, data missing |
| fig2-single-neuron-responses | blocked, data missing |
| fig3abcd-single-electrode-datasets | blocked, data missing |
| fig4-emg-no-rotation | blocked, data missing |
| fig5-generator-model | blocked, data missing |
| fig6cd-speed-independent-frequency | blocked, data missing |

How the analysis differs from the paper's. The Units tables hold unsorted threshold crossings, 192 per session (one per electrode), where the paper analysed 146 and 218 sorted single and multi-unit isolations; every jPCA claim is judged on these units, as the claims file says. The paper's custom smoothing step (Methods: it "had essentially no effect on any of the population-level analyses") is not applied. Rates are aligned on movement onset only, not also on target onset and the go cue. Which movement-onset column the paper's alignment corresponds to, and over which window the normalisation range is taken, are not stated; both are run under each reading, and the offline onset with the range over the analysed window is the main reading. The simulated velocity and complex kinematic populations are built here from the Methods' description, with a lead of the rates over the hand (100 ms) and a time course of the preparatory term (falling as the movement term starts) that the Methods do not give; the outcomes of both model claims change with these choices.

What was found about the dataset. All ten files of the version were read. The paper's two array sessions are there with 108 conditions each; the Units tables are unsorted threshold crossings, which the files state. Two of the eight further sessions (sub-Nitschke ses-20100920 and ses-20100922) have no `maze_condition`, `correct_reach` or `trial_version` columns. No recording is deposited twice (all trial-interval and spike-count hashes differ). Hand position is in mm (with a conversion to metres), its speed rises at `offline_move_onset_time`, and the hand ends about 9 to 11 mm from the hit target minus the cursor offset. Reach direction decodes at chance from both arrays before target onset and above chance after it. The lag between the arrays' movement-related changes is constant over each session (-4 to 11 ms in J-array and 25 to 32 ms in N-array over six blocks of the selected trials, M1 later), so there is no drift of one array's spikes against the other. After target onset the arrays respond within 9 ms of each other in N-array (half-rise of the across-condition spread at 99 and 101 ms) and -4 ms in J-array, so a fixed offset of 25 ms or more between the arrays' clocks is not supported; the later M1 change around movement (and after the go cue, 60 ms in N-array) is in the responses.

In [ ]:
print(f"notebook ran in {(time.time() - T_START) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")